# Nutrition5k — Experiment 3: Depth as 4th Channel

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, sys

if os.path.isdir('/content/Nutrition5k'):
    subprocess.run(['git', '-C', '/content/Nutrition5k', 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/T0MYYY/Nutrition5k.git', '/content/Nutrition5k', '-b', 'master', '-q'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/Nutrition5k', '-q'], check=True)

if '/content/Nutrition5k' not in sys.path:
    sys.path.insert(0, '/content/Nutrition5k')
import importlib; importlib.invalidate_caches()

CONFIG_DIR = '/content/Nutrition5k/configs/colab'
print(f'Ready. Config: {CONFIG_DIR}')

## GPU Check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory // 1024**3, 'GB')

## Adjust Batch Size

In [ ]:
import torch, yaml

cfg_path = f'{CONFIG_DIR}/exp3_depth_channel.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory // 1024**3
    if vram_gb >= 80:   bs = 256
    elif vram_gb >= 40: bs = 128
    elif vram_gb >= 24: bs = 64
    else:               bs = 32
    scale = bs / 256
    cfg['training']['batch_size'] = bs
    cfg['training']['lr']      = round(3e-4 * scale, 7)
    cfg['training']['head_lr'] = round(1e-3 * scale, 7)

with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f)
print(f"batch={cfg['training']['batch_size']}  lr={cfg['training']['lr']:.2e}  head_lr={cfg['training']['head_lr']:.2e}")

## Load data into RAM

First run: parallel Drive read + zip to Drive. Next runs: zip→RAM in seconds.

In [ ]:
import os, subprocess, threading

subprocess.run(['apt-get', 'install', '-qq', '-y', 'zstd'],
               check=True, capture_output=True)

DRIVE  = '/content/drive/MyDrive/nutrition5k/data'
DST    = '/dev/shm'
marker = '/dev/shm/realsense_overhead'

def _extract(archive):
    src = f'{DRIVE}/{archive}'
    print(f'  Extracting {archive} ...')
    subprocess.run(f'zstd -d --stdout "{src}" | tar -x -C {DST}',
                   shell=True, check=True)
    print(f'  Done: {archive}')

os.makedirs(marker, exist_ok=True)
n_existing = sum(1 for d in os.scandir(marker) if d.is_dir())
todo = []
if n_existing <= 50:
    todo = ['depth_train.tar.zst', 'depth_test.tar.zst']

if todo:
    threads = [threading.Thread(target=_extract, args=(a,), daemon=True) for a in todo]
    for t in threads: t.start()
    for t in threads: t.join()
else:
    print('Already extracted.')

n = sum(1 for d in os.scandir(marker) if d.is_dir())
print(f'Ready: {n} dishes in {marker}')

## Train

In [ ]:
from nutrition5k_pkg.train import run
best_ckpt = run(f'{CONFIG_DIR}/exp3_depth_channel.yaml')
print('Best checkpoint:', best_ckpt)

## Evaluate on Nutri-Test

In [ ]:
import torch
from nutrition5k_pkg.train import load_config, _build_model
from nutrition5k_pkg.evaluate import evaluate

cfg = load_config(f'{CONFIG_DIR}/exp3_depth_channel.yaml')
model = _build_model(cfg)
model.load_state_dict(torch.load('/content/drive/MyDrive/nutrition5k/checkpoints/exp3/best.pt', map_location='cpu'))
results_dir = '/content/drive/MyDrive/nutrition5k/results/exp3'
results = evaluate(model, f'{CONFIG_DIR}/exp3_depth_channel.yaml', results_dir)
print('Results saved to:', results_dir)

In [ ]:
from google.colab import runtime
runtime.unassign()